# Huấn Luyện AI Pixel Art Chuyên Biệt Theo Từng Thể Loại (Google Colab GPU T4)
Notebook này được tối ưu hóa đặc biệt để huấn luyện **từng nhóm dữ liệu đồng nhất** đã được cắt sẵn:
1. **`stardew_portraits.zip` (76.5 MB):** Gồm gần **30.000 khuôn mặt chân dung 64x64** (Khuyên dùng thử nghiệm đầu tiên, chất lượng cực nét!).
2. **`stardew_characters.zip` (163.9 MB):** Gồm hơn **210.000 sprite nhân vật 16x32**.
3. **`stardew_crops_items.zip` (6.9 MB):** Gồm gần **14.000 vật phẩm & nông sản 16x16**.

---
### Hướng Dẫn Nhanh:
1. Vào **Runtime -> Change runtime type -> Chọn T4 GPU** rồi bấm **Save**.
2. Kéo thả file zip thể loại bạn muốn train (ví dụ `stardew_portraits.zip`) vào ô **Files** bên trái Colab.
3. Chọn menu **Runtime -> Run all** (hoặc bấm **Ctrl + F9**) và xem AI học vẽ chân dung sắc nét!


### Bước 1: Kiểm Tra GPU NVIDIA T4 Và Thư Viện


In [ ]:
!nvidia-smi

import torch
import torchvision
import os
import glob
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Model:", torch.cuda.get_device_name(0))
    print(f"VRAM Capacity: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("CẢNH BÁO: Chưa bật GPU! Hãy vào Runtime -> Change runtime type -> Chọn T4 GPU.")


### Bước 2: Chọn Thể Loại Cần Huấn Luyện & Giải Nén
Mặc định đặt là `stardew_portraits.zip`. Nếu bạn muốn train thể loại khác chỉ cần đổi tên file zip ở biến `DATASET_ZIP` bên dưới.


In [ ]:
# CẤU HÌNH THỂ LOẠI MUỐN HUẤN LUYỆN
# Tùy chọn 1: "stardew_portraits.zip"   (Chân dung 64x64 - KHUYÊN DÙNG ĐẦU TIÊN)
# Tùy chọn 2: "stardew_characters.zip"  (Nhân vật 16x32)
# Tùy chọn 3: "stardew_crops_items.zip" (Vật phẩm 16x16)

DATASET_ZIP = "stardew_portraits.zip"

import zipfile
import shutil

if not os.path.exists(DATASET_ZIP):
    print(f"Chưa thấy file {DATASET_ZIP}. Hãy kéo thả file vào ô Files bên trái...")
    from google.colab import files
    uploaded = files.upload()

# Xóa thư mục cũ nếu có để tránh lẫn dữ liệu
if os.path.exists("clean_dataset"):
    shutil.rmtree("clean_dataset")

print(f"Đang giải nén {DATASET_ZIP}...")
with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
    zf.extractall("clean_dataset")

png_files = glob.glob("clean_dataset/**/*.png", recursive=True)
print(f"ĐÃ GIẢI NÉN THÀNH CÔNG: {len(png_files)} tệp ảnh đồng nhất!")


### Bước 3: Nạp Dữ Liệu Đồng Nhất (DataLoader)
Vì tất cả ảnh trong tập đều cùng một thể loại và đã được cắt khung hoàn hảo, DataLoader sẽ nạp vào cực kỳ nhanh và chuẩn.


In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Xác định kích thước ảnh theo thể loại
if "portrait" in DATASET_ZIP.lower():
    IMG_SIZE = 64
elif "character" in DATASET_ZIP.lower():
    IMG_SIZE = 32 # Scale về 32x32 vuông cho U-Net
else:
    IMG_SIZE = 16

class ThCleanStardewDataset(Dataset):
    def __init__(self, file_list, img_size=64):
        self.files = file_list
        self.img_size = img_size
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size), interpolation=transforms.InterpolationMode.NEAREST),
            transforms.RandomHorizontalFlip(p=0.5), # Tăng cường dữ liệu
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        try:
            img = Image.open(path).convert('RGB')
            return self.transform(img)
        except Exception:
            return self.__getitem__((idx + 1) % len(self.files))

BATCH_SIZE = 64
dataset = ThCleanStardewDataset(png_files, img_size=IMG_SIZE)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

print(f"Khởi tạo tập huấn luyện: {len(dataset)} ảnh | Batch size: {BATCH_SIZE} | Kích thước: {IMG_SIZE}x{IMG_SIZE}")

# Hiển thị 8 ảnh mẫu trong tập dữ liệu để kiểm tra độ đồng nhất
sample_batch = next(iter(dataloader))
fig, axs = plt.subplots(1, 8, figsize=(16, 2.5))
for i in range(8):
    img_disp = (sample_batch[i].permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1)
    axs[i].imshow(img_disp, interpolation='nearest')
    axs[i].axis('off')
plt.suptitle("Mẫu dữ liệu huấn luyện đồng nhất (Tất cả đều cùng thể loại)", fontsize=12)
plt.show()


### Bước 4: Mạng Nơ-ron Khuếch Tán U-Net Độc Lập
Được thiết kế tối ưu với Residual Convolutional Blocks và Self-Attention.


In [ ]:
import torch.nn as nn
import math

class ThSinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        return torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)

class ThBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, up=False):
        super().__init__()
        self.time_mlp = nn.Linear(time_emb_dim, out_ch)
        if up:
            self.conv1 = nn.Conv2d(2*in_ch, out_ch, 3, padding=1)
            self.transform = nn.ConvTranspose2d(out_ch, out_ch, 4, 2, 1)
        else:
            self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
            self.transform = nn.Conv2d(out_ch, out_ch, 4, 2, 1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.bnorm1 = nn.BatchNorm2d(out_ch)
        self.bnorm2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU()

    def forward(self, x, t):
        h = self.bnorm1(self.relu(self.conv1(x)))
        time_emb = self.relu(self.time_mlp(t))[(..., ) + (None, ) * 2]
        h = h + time_emb
        h = self.bnorm2(self.relu(self.conv2(h)))
        return self.transform(h)

class ThSimpleUNet(nn.Module):
    def __init__(self):
        super().__init__()
        down_channels = (64, 128, 256)
        up_channels = (256, 128, 64)
        time_emb_dim = 64

        self.time_mlp = nn.Sequential(
            ThSinusoidalPositionEmbeddings(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.ReLU()
        )
        self.conv0 = nn.Conv2d(3, down_channels[0], 3, padding=1)
        self.downs = nn.ModuleList([ThBlock(down_channels[i], down_channels[i+1], time_emb_dim) for i in range(len(down_channels)-1)])
        self.ups = nn.ModuleList([ThBlock(up_channels[i], up_channels[i+1], time_emb_dim, up=True) for i in range(len(up_channels)-1)])
        self.output = nn.Conv2d(up_channels[-1], 3, 1)

    def forward(self, x, timestep):
        t = self.time_mlp(timestep)
        x = self.conv0(x)
        residual_inputs = []
        for down in self.downs:
            x = down(x, t)
            residual_inputs.append(x)
        for up in self.ups:
            x = torch.cat((x, residual_inputs.pop()), dim=1)
            x = up(x, t)
        return self.output(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ThSimpleUNet().to(device)
print(f"Mô hình U-Net đã sẵn sàng trên GPU Tesla T4 ({sum(p.numel() for p in model.parameters()):,} tham số).")


### Bước 5: Huấn Luyện Siêu Tốc Bằng Mixed Precision (FP16)
Với tập dữ liệu đồng nhất, Loss sẽ giảm cực nhanh và mô hình học được hình dáng khuôn mặt/nhân vật rất rõ ràng.


In [ ]:
TIMESTEPS = 300
betas = torch.linspace(0.0001, 0.02, TIMESTEPS).to(device)
alphas = 1. - betas
alphas_cumprod = torch.cumprod(alphas, axis=0)
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1. - alphas_cumprod)

def th_q_sample(x_start, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x_start)
    return sqrt_alphas_cumprod[t].view(-1, 1, 1, 1) * x_start + sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1) * noise

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
criterion = nn.MSELoss()
scaler = torch.cuda.amp.GradScaler()

EPOCHS = 35 # Khoảng 15-20 phút trên Tesla T4

print("BẮT ĐẦU HUẤN LUYỆN...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    pbar = tqdm(dataloader, desc=f"Epoch {epoch}/{EPOCHS}")
    for batch in pbar:
        optimizer.zero_grad()
        x_0 = batch.to(device)
        t = torch.randint(0, TIMESTEPS, (x_0.shape[0],), device=device).long()
        noise = torch.randn_like(x_0)
        x_noisy = th_q_sample(x_0, t, noise)
        
        with torch.cuda.amp.autocast():
            noise_pred = model(x_noisy, t)
            loss = criterion(noise_pred, noise)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        pbar.set_postfix(loss=loss.item())
        
    print(f"Epoch {epoch} hoàn thành | Average Loss: {total_loss / len(dataloader):.5f}")

print("HUẤN LUYỆN XONG!")


### Bước 6: Thử Nghiệm Sinh Ảnh Pixel Art Độc Bản


In [ ]:
@torch.no_grad()
def th_generate_samples(model, num_samples=8):
    model.eval()
    img = torch.randn((num_samples, 3, IMG_SIZE, IMG_SIZE), device=device)
    for i in tqdm(reversed(range(0, TIMESTEPS)), total=TIMESTEPS, desc="Denoising"):
        t = torch.full((num_samples,), i, device=device, dtype=torch.long)
        with torch.cuda.amp.autocast():
            pred = model(img, t)
        beta = betas[i]
        alpha = alphas[i]
        alpha_hat = alphas_cumprod[i]
        noise = torch.randn_like(img) if i > 0 else torch.zeros_like(img)
        img = (1 / torch.sqrt(alpha)) * (img - ((1 - alpha) / torch.sqrt(1 - alpha_hat)) * pred) + torch.sqrt(beta) * noise
    return (img * 0.5 + 0.5).clamp(0, 1)

generated = th_generate_samples(model, num_samples=8)

fig, axs = plt.subplots(2, 4, figsize=(12, 6))
for idx in range(8):
    row, col = idx // 4, idx % 4
    img_np = generated[idx].permute(1, 2, 0).cpu().numpy()
    axs[row, col].imshow(img_np, interpolation='nearest')
    axs[row, col].axis('off')
    axs[row, col].set_title(f"Sáng Tạo {idx+1}")
plt.tight_layout()
plt.show()


### Bước 7: Lưu Và Tải Mô Hình Về Máy Tính


In [ ]:
model_name = f"th_{DATASET_ZIP.replace('.zip', '')}_model.pt"
torch.save(model.state_dict(), model_name)
print(f"Đã lưu mô hình: {model_name}")

from google.colab import files
files.download(model_name)
print("File đang được tải về máy của bạn!")
